In [1]:
!pwd

/home/cs_unix/docs/compvision/diabetes-retinopathy


In [ ]:
%matplotlib inline

# attempt_24.ipynb -- Diabetic Retinopathy Detection

**Changes vs attempt_23.ipynb:**
- **(C-7) Load pre-trained backbones**: all four FT backbones (DenseNet121,
  DenseNet169, ResNeXt50, SE-ResNeXt50) are loaded from `.pth` files trained
  in attempt_23 — no retraining. Only CustomNetV2 is trained from scratch.
- **(C-8) MODELS_PATH / SAVE_PATH constants**: all file I/O goes through these
  two variables, set once at the top of the notebook.
- **(C-9) Reduced batch sizes**: TRAIN_BATCH=32, VAL_BATCH=128 (safer at 380px).
- **(C-10) Memory management**: FT models kept on CPU; moved to GPU one at a time
  during val and test scoring to avoid CUDA OOM.
- All transforms (380px, ±180° rotation), ensemble search, and CustomNetV2
  unchanged from attempt_23.


## 1. Imports & Setup

In [ ]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, utils, models
from torchvision.models import densenet121, DenseNet121_Weights
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings
import itertools

warnings.filterwarnings('ignore')
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

plt.ion()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# CSVs and images/ folder live at the project root
DATA_ROOT   = '/kaggle/input/datasets/mariamuozperez/lab5-cv'

# Directory containing pre-trained .pth files from attempt_23
MODELS_PATH = '/kaggle/working'

# Directory for new .pth checkpoints and submission outputs
SAVE_PATH   = '/kaggle/working'

TRAIN_BATCH = 32    # for CustomNetV2 training
VAL_BATCH   = 128   # for val/test scoring at 380px


In [ ]:
# Run once to extract data, then comment out
# import zipfile
# with zipfile.ZipFile('./db.zip', 'r') as z:
#     z.extractall('./data')

## 2. Dataset

In [ ]:
class RetinopathyDataset(Dataset):
    """Retinopathy dataset."""

    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.levels  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        self.classes = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Transforms

In [ ]:
class CropByEye(object):
    """Threshold-based eye segmentation and crop."""
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    """Local contrast normalisation for fundus images (Ben Graham, Kaggle DR 2015).
    Subtracts a Gaussian-blurred background to enhance microaneurysms and exudates.
    sigmaX controls the background scale (approx. major vessel diameter)."""
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        # Accept both uint8 [0,255] (from CropByEye) and float [0,1]
        if image.dtype == np.uint8:
            img_u8 = image
        else:
            img_u8 = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        # Circular mask: set background outside the eye to neutral gray (128)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    """Rescale image to output_size (int = shortest side; tuple = exact)."""
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    """Random crop to output_size."""
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    """Centre crop to output_size."""
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    """Convert HxWxC numpy array to CxHxW torch tensor."""
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    """Normalise per-channel using mean and std."""
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


class TVCenterCrop(object):
    """Centre crop via torchvision (numpy->PIL->op->numpy)."""
    def __init__(self, size):
        self.CC = transforms.CenterCrop(size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.CC(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomHorizontalFlip(object):
    """Random horizontal flip (simulates left/right eye symmetry)."""
    def __init__(self, p=0.5):
        self.flip = transforms.RandomHorizontalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    """Random rotation +/-degrees (simulates camera tilt)."""
    def __init__(self, degrees=15):
        self.rotate = transforms.RandomRotation(degrees=degrees)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.rotate(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    """Random colour jitter (simulates cross-site imaging variation)."""
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05):
        self.jitter = transforms.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.jitter(pil)))
        return {'image': image, 'eye': eye, 'label': label}

## 4. Data Pipelines & DataLoaders

In [ ]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

# Training: augmentation pipeline — 380 px, full 360° rotation
train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(416),
    TVRandomHorizontalFlip(p=0.5),
    TVRandomRotation(degrees=180),
    TVColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    RandomCrop(380),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

# Val / test: Rescale(416) then CenterCrop(380)
eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(416),
    CenterCrop(380),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

# maxSize=0 uses all 2000 training images
train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'train.csv'),
    root_dir=DATA_ROOT,
    maxSize=0,
    transform=train_transform)

val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT,
    transform=eval_transform)

test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT,
    transform=eval_transform)

print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')


In [ ]:
# Sanity check: verify BenGraham is not producing all-gray images
# Expected: mean near 0 (after ImageNet normalization), NOT near 0.5
_sample = train_dataset[0]
_img = _sample["image"].numpy().transpose(1, 2, 0)
print(f"Pipeline output -- dtype: {_img.dtype}, "
      f"min: {_img.min():.3f}, max: {_img.max():.3f}, mean: {_img.mean():.3f}")
assert abs(_img.mean()) < 0.3, (
    f"BenGraham all-gray bug still active! mean={_img.mean():.3f} "
    "(expected near 0 after ImageNet norm, not 0.5)"
)
print("Sanity check passed.")


In [ ]:
# Class-balanced sampler: oversample DR to stabilise CustomNetV2 training from scratch
train_labels_bin_for_sampler = (train_dataset.dataset['label'].values > 0).astype(int)
class_counts  = np.bincount(train_labels_bin_for_sampler)          # [n_neg, n_pos]
sample_weights = np.where(train_labels_bin_for_sampler == 1,
                           1.0 / class_counts[1],
                           1.0 / class_counts[0])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset),
    replacement=True,
)

# num_workers=0 required on Windows (PyTorch multiprocessing bug)
# shuffle=False when using a sampler (they are mutually exclusive)
train_dataloader = DataLoader(train_dataset, batch_size=TRAIN_BATCH, sampler=sampler,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=VAL_BATCH,   shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=VAL_BATCH,   shuffle=False, num_workers=0)

# Class-weighted loss: pos_weight = n_neg / n_pos handles the 73%/27% imbalance
train_labels_bin = (train_dataset.dataset['label'].values > 0).astype(int)
n_neg = int((train_labels_bin == 0).sum())
n_pos = int((train_labels_bin == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f'No-DR: {n_neg}  DR: {n_pos}  pos_weight: {pos_weight.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

image_datasets = {'train': train_dataset, 'val': val_dataset}
dataloaders    = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes  = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names    = train_dataset.classes

## 5. Training & Evaluation Utilities

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7, label_smoothing=0.0):
    """Train with early stopping on val AUC. Returns best-weight model.
    label_smoothing > 0 softens binary targets for training batches only."""
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auc   = 0.0
    best_epoch = -1
    no_improve = 0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == "train" else model.eval()

            numSamples   = dataset_sizes[phase]
            outputs_m    = np.zeros((numSamples,), dtype=float)
            labels_m     = np.zeros((numSamples,), dtype=int)
            running_loss = 0.0
            contSamples  = 0

            for sample in dataloaders[phase]:
                inputs = sample['image'].to(device).float()
                labels = sample['label'].to(device).float()
                batchSize = labels.shape[0]
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(inputs).flatten()
                    if label_smoothing > 0.0 and phase == 'train':
                        labels_ls = labels * (1 - label_smoothing) + label_smoothing / 2.0
                        loss = criterion(logits, labels_ls)
                    else:
                        loss = criterion(logits, labels)
                    scores = torch.sigmoid(logits).detach()
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * batchSize
                outputs_m[contSamples:contSamples + batchSize] = scores.cpu().numpy()
                labels_m [contSamples:contSamples + batchSize] = labels.cpu().numpy()
                contSamples += batchSize

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print('{} Loss: {:.4f}  AUC: {:.4f}'.format(phase, epoch_loss, epoch_auc))

            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc       = epoch_auc
                    best_epoch     = epoch
                    best_model_wts = copy.deepcopy(model.state_dict())
                    no_improve     = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f'Early stopping: no improvement for {patience} epochs.')
                        model.load_state_dict(best_model_wts)
                        return model
        print()

    elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(elapsed // 60, elapsed % 60))
    print('Best model: epoch {:d}  val AUC: {:.4f}'.format(best_epoch, best_auc))
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
def eval_val_auc(model, name, tta=False):
    """Run model on validation set and print AUC.
    tta=True: 4-pass TTA (original + hflip + vflip + rot180)."""
    model.eval()
    n        = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,),   dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1  = torch.sigmoid(model(inputs))
                s2  = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3  = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4  = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]    = sample['label'].numpy()
            cont += bs
    auc    = metrics.roc_auc_score(labels_m, scores_m)
    suffix = ' (TTA-4)' if tta else ''
    print(f'{name}{suffix}  --  val AUC: {auc:.4f}')
    return auc


def test_model(model, tta=False):
    """Run model on test set. Returns (1000, 1) score array.
    tta=True: 4-pass TTA (original + hflip + vflip + rot180)."""
    model.eval()
    numSamples = len(test_dataset)
    outputs_m  = np.zeros((numSamples, 1), dtype=float)
    contSamples = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1  = torch.sigmoid(model(inputs))
                s2  = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3  = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4  = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            outputs_m[contSamples:contSamples + bs, :] = out.cpu().numpy()
            contSamples += bs
    return outputs_m

---
## 6. CustomNetV2 (CUSTOM category)

**Key improvements over old CustomNet:**
- 3Ã—3 same-padding convs (vs 5Ã—5 no-pad) â€” preserves spatial info; only MaxPool shrinks dims
- Channels: 3â†’32â†’64â†’128â†’256â†’256 (vs 3â†’6â†’16â†’32â†’64) â€” wider filter banks
- **Global Average Pooling** replaces 6400-dim flatten â€” classifier params drop from ~771k to ~33k
- `bias=False` in Conv2d â€” BN's Î² absorbs bias, removes redundancy
- AdamW + CosineAnnealingLR (vs SGD + StepLR)

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block (Hu et al., 2017).

    Channel-wise attention: GAP each channel to a scalar, pass through a small
    bottleneck MLP, sigmoid, and rescale the input feature map.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)  # guard against tiny bottlenecks
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.squeeze(x).view(b, c)
        w = self.excitation(w).view(b, c, 1, 1)
        return x * w


class CustomNetV2(nn.Module):
    """
    5-block CNN with 3x3 same-convolutions, BatchNorm, GAP, and SE blocks in
    the three deepest stages (channels 128, 256, 256).
    No pretrained weights -- valid for the CUSTOM Codabench category.

    SE placement: after conv+BN+ReLU, before MaxPool.
    Blocks 1-2 (32, 64 ch): no SE (low-level features).
    Blocks 3-5 (128, 256, 256 ch): SE with reduction=16.

    Spatial progression:
      224 -> 112 -> 56 -> 28 -> 14 -> 7
    GAP: 256 x 7 x 7 -> 256 scalars
    """
    def __init__(self, se_reduction=16):
        super().__init__()

        def _block(cin, cout, use_se=False):
            layers = [
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
            ]
            if use_se:
                layers.append(SEBlock(cout, reduction=se_reduction))
            layers.append(nn.MaxPool2d(2, 2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            _block(3,   32,  use_se=False),  # 224 -> 112
            _block(32,  64,  use_se=False),  # 112 ->  56
            _block(64,  128, use_se=True ),  #  56 ->  28
            _block(128, 256, use_se=True ),  #  28 ->  14
            _block(256, 256, use_se=True ),  #  14 ->   7
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

In [ ]:
# Sanity check: verify output shape and count parameters
_net = CustomNetV2().to(device)
_inp = next(iter(train_dataloader))['image'].to(device).float()
with torch.no_grad():
    _out = _net(_inp)
print(f'Input shape:  {_inp.shape}')
print(f'Output shape: {_out.shape}')

total     = sum(p.numel() for p in _net.parameters())
clf       = sum(p.numel() for p in _net.classifier.parameters())
se_params = sum(p.numel() for n, p in _net.named_parameters() if 'excitation' in n)
print(f'CustomNetV2 -- total: {total:,}  classifier: {clf:,}  SE-only: {se_params:,}')
del _net, _inp, _out

In [ ]:
# Reset all seeds before model init for reproducibility
random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

customNetV2 = CustomNetV2().to(device)

# AdamW + CosineAnnealingLR: converges reliably on small datasets
optimizer_custom  = optim.AdamW(customNetV2.parameters(), lr=1e-3, weight_decay=5e-3)
scheduler_custom  = lr_scheduler.CosineAnnealingLR(optimizer_custom, T_max=50)

In [ ]:
customNetV2 = train_model(customNetV2, criterion, optimizer_custom, scheduler_custom,
                          num_epochs=50, patience=10)

In [ ]:
torch.save(customNetV2.state_dict(), os.path.join(SAVE_PATH, 'best_customnetv2_se.pth'))
print(f'Saved: {os.path.join(SAVE_PATH, "best_customnetv2_se.pth")}')
auc_custom = eval_val_auc(customNetV2, 'CustomNetV2 + SE')

# Fine-tuning category

In [ ]:
!pip install timm

In [ ]:
from torchvision.models import (
    densenet121, DenseNet121_Weights,
    densenet169, DenseNet169_Weights,
    resnext50_32x4d, ResNeXt50_32X4D_Weights,
)
import timm

In [ ]:
# Reproducibility helper
def set_seed(seed=42):
    random.seed(seed)
    npr.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Model = ensemble between DenseNet121, DenseNet159, ResNext50, SeResNext50

In [ ]:
def build_ft_model(model_name, device):
    """
    Returns:
        model
        stage1_groups_fn(model) -> list of parameter groups for optimizer
        stage2_groups_fn(model) -> list of parameter groups for optimizer
        stage1_unfreeze_fn(model)
        stage2_unfreeze_fn(model)
    """

    if model_name == 'densenet121':
        model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)

        for p in model.parameters():
            p.requires_grad = False

        model.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(1024, 1)
        )

        def stage1_unfreeze_fn(m):
            for p in m.features.denseblock4.parameters():
                p.requires_grad = True
            for p in m.features.norm5.parameters():
                p.requires_grad = True

        def stage2_unfreeze_fn(m):
            for p in m.features.denseblock3.parameters():
                p.requires_grad = True
            for p in m.features.transition3.parameters():
                p.requires_grad = True

        def stage1_groups_fn(m):
            return [
                {'params': m.features.denseblock4.parameters(), 'lr': 3e-4},
                {'params': m.features.norm5.parameters(),       'lr': 3e-4},
                {'params': m.classifier.parameters(),           'lr': 3e-4},
            ]

        def stage2_groups_fn(m):
            return [
                {'params': m.features.denseblock3.parameters(), 'lr': 3e-5},
                {'params': m.features.transition3.parameters(), 'lr': 3e-5},
                {'params': m.features.denseblock4.parameters(), 'lr': 3e-5},
                {'params': m.features.norm5.parameters(),       'lr': 3e-5},
                {'params': m.classifier.parameters(),           'lr': 1e-4},
            ]

    elif model_name == 'densenet169':
        model = densenet169(weights=DenseNet169_Weights.IMAGENET1K_V1)

        for p in model.parameters():
            p.requires_grad = False

        model.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(1664, 1)
        )

        def stage1_unfreeze_fn(m):
            for p in m.features.denseblock4.parameters():
                p.requires_grad = True
            for p in m.features.norm5.parameters():
                p.requires_grad = True

        def stage2_unfreeze_fn(m):
            for p in m.features.denseblock3.parameters():
                p.requires_grad = True
            for p in m.features.transition3.parameters():
                p.requires_grad = True

        def stage1_groups_fn(m):
            return [
                {'params': m.features.denseblock4.parameters(), 'lr': 3e-4},
                {'params': m.features.norm5.parameters(),       'lr': 3e-4},
                {'params': m.classifier.parameters(),           'lr': 3e-4},
            ]

        def stage2_groups_fn(m):
            return [
                {'params': m.features.denseblock3.parameters(), 'lr': 2e-5},
                {'params': m.features.transition3.parameters(), 'lr': 2e-5},
                {'params': m.features.denseblock4.parameters(), 'lr': 2e-5},
                {'params': m.features.norm5.parameters(),       'lr': 2e-5},
                {'params': m.classifier.parameters(),           'lr': 1e-4},
            ]

    elif model_name == 'resnext50':
        model = resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V2)

        for p in model.parameters():
            p.requires_grad = False

        in_feats = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_feats, 1)
        )

        def stage1_unfreeze_fn(m):
            for p in m.layer4.parameters():
                p.requires_grad = True

        def stage2_unfreeze_fn(m):
            for p in m.layer3.parameters():
                p.requires_grad = True

        def stage1_groups_fn(m):
            return [
                {'params': m.layer4.parameters(), 'lr': 3e-4},
                {'params': m.fc.parameters(),     'lr': 3e-4},
            ]

        def stage2_groups_fn(m):
            return [
                {'params': m.layer3.parameters(), 'lr': 2e-5},
                {'params': m.layer4.parameters(), 'lr': 3e-5},
                {'params': m.fc.parameters(),     'lr': 1e-4},
            ]

    elif model_name == 'seresnext50':
        # Requires timm
        model = timm.create_model('seresnext50_32x4d', pretrained=True, num_classes=1)

        for p in model.parameters():
            p.requires_grad = False

        # timm head naming differs by model
        if hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
            in_feats = model.fc.in_features
            model.fc = nn.Sequential(
                nn.Dropout(p=0.5),
                nn.Linear(in_feats, 1)
            )
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Linear):
            in_feats = model.classifier.in_features
            model.classifier = nn.Sequential(
                nn.Dropout(p=0.5),
                nn.Linear(in_feats, 1)
            )
            head_params = model.classifier.parameters()
        else:
            raise ValueError("Unsupported SE-style model head structure")

        # ResNet-like stage names in timm SE-ResNeXt
        def stage1_unfreeze_fn(m):
            for p in m.layer4.parameters():
                p.requires_grad = True

        def stage2_unfreeze_fn(m):
            for p in m.layer3.parameters():
                p.requires_grad = True

        def stage1_groups_fn(m):
            return [
                {'params': m.layer4.parameters(), 'lr': 3e-4},
                {'params': head_params,           'lr': 3e-4},
            ]

        def stage2_groups_fn(m):
            return [
                {'params': m.layer3.parameters(), 'lr': 2e-5},
                {'params': m.layer4.parameters(), 'lr': 3e-5},
                {'params': head_params,           'lr': 1e-4},
            ]

    else:
        raise ValueError(f'Unknown model_name: {model_name}')

    model = model.to(device)
    return model, stage1_groups_fn, stage2_groups_fn, stage1_unfreeze_fn, stage2_unfreeze_fn

In [ ]:
# Generic validation score extractor
def get_val_scores(model, val_dataloader, device, tta=False):
    model.eval()
    n = len(val_dataloader.dataset)
    scores = np.zeros((n, 1), dtype=np.float32)
    labels = np.zeros((n,), dtype=np.int64)

    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]

            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))      # hflip
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))      # vflip
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))   # rot180
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))

            scores[cont:cont+bs, :] = out.detach().cpu().numpy()
            labels[cont:cont+bs] = sample['label'].numpy()
            cont += bs

    return scores, labels

In [ ]:
# Generic training
def train_ft_backbone(
    model_name,
    criterion,
    train_model,
    eval_val_auc,
    val_dataloader,
    device,
    stage2_gate_auc=0.752,
    seed=42
):
    """
    Trains Stage 1, optionally Stage 2, returns:
        best_model, info_dict
    """
    print(f'\n{"="*80}\nTraining backbone: {model_name}\n{"="*80}')
    set_seed(seed)

    model, stage1_groups_fn, stage2_groups_fn, stage1_unfreeze_fn, stage2_unfreeze_fn = build_ft_model(model_name, device)

    # ---- Stage 1 ----
    stage1_unfreeze_fn(model)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'{model_name} Stage 1 trainable: {trainable:,} / {total:,}')

    optimizer_s1 = optim.AdamW(stage1_groups_fn(model), weight_decay=1e-2)
    scheduler_s1 = lr_scheduler.CosineAnnealingLR(optimizer_s1, T_max=30)

    model_s1 = train_model(
        model, criterion, optimizer_s1, scheduler_s1,
        num_epochs=30, patience=7, label_smoothing=0.05
    )

    stage1_path = os.path.join(SAVE_PATH, f'best_{model_name}_s1.pth')
    torch.save(model_s1.state_dict(), stage1_path)
    print(f'Saved: {stage1_path}')

    auc_s1 = eval_val_auc(model_s1, f'{model_name} Stage 1', tta=False)
    auc_s1_tta = eval_val_auc(model_s1, f'{model_name} Stage 1', tta=True)

    best_model = copy.deepcopy(model_s1)
    best_auc_tta = auc_s1_tta
    best_stage = 's1'

    # ---- Stage 2 gate ----
    if auc_s1_tta < stage2_gate_auc:
        print(f'{model_name}: Stage 1 TTA AUC {auc_s1_tta:.4f} < gate {stage2_gate_auc:.4f}. Skipping Stage 2.')
        return best_model, {
            'model_name': model_name,
            'best_stage': best_stage,
            'auc_s1': auc_s1,
            'auc_s1_tta': auc_s1_tta,
            'auc_s2_tta': None,
            'best_auc_tta': best_auc_tta,
            'stage1_path': stage1_path,
            'stage2_path': None,
        }

    # ---- Stage 2 ----
    print(f'{model_name}: Stage 1 passed gate. Running Stage 2...')
    model_s2, _, stage2_groups_fn, _, stage2_unfreeze_fn = build_ft_model(model_name, device)
    model_s2.load_state_dict(torch.load(stage1_path, map_location=device))
    stage1_unfreeze_fn(model_s2)
    stage2_unfreeze_fn(model_s2)

    trainable_s2 = sum(p.numel() for p in model_s2.parameters() if p.requires_grad)
    print(f'{model_name} Stage 2 trainable: {trainable_s2:,}')

    optimizer_s2 = optim.AdamW(stage2_groups_fn(model_s2), weight_decay=1e-2)

    warmup_scheduler = lr_scheduler.LinearLR(
        optimizer_s2, start_factor=0.1, end_factor=1.0, total_iters=3
    )
    cosine_scheduler = lr_scheduler.CosineAnnealingLR(
        optimizer_s2, T_max=12
    )
    scheduler_s2 = lr_scheduler.SequentialLR(
        optimizer_s2,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[3]
    )

    model_s2 = train_model(
        model_s2, criterion, optimizer_s2, scheduler_s2,
        num_epochs=15, patience=5, label_smoothing=0.05
    )

    auc_s2_tta = eval_val_auc(model_s2, f'{model_name} Stage 2', tta=True)

    stage2_path = os.path.join(SAVE_PATH, f'best_{model_name}_s2.pth')
    if auc_s2_tta > auc_s1_tta:
        print(f'{model_name}: Stage 2 better ({auc_s2_tta:.4f} > {auc_s1_tta:.4f})')
        torch.save(model_s2.state_dict(), stage2_path)
        print(f'Saved: {stage2_path}')
        best_model = copy.deepcopy(model_s2)
        best_auc_tta = auc_s2_tta
        best_stage = 's2'
    else:
        print(f'{model_name}: Stage 2 did NOT improve ({auc_s2_tta:.4f} <= {auc_s1_tta:.4f}), keeping Stage 1')
        stage2_path = None

    return best_model, {
        'model_name': model_name,
        'best_stage': best_stage,
        'auc_s1': auc_s1,
        'auc_s1_tta': auc_s1_tta,
        'auc_s2_tta': auc_s2_tta,
        'best_auc_tta': best_auc_tta,
        'stage1_path': stage1_path,
        'stage2_path': stage2_path,
    }

In [ ]:
def load_ft_model(model_name, models_dir, device):
    """Build architecture and load best available checkpoint from models_dir."""
    s2 = os.path.join(models_dir, f'best_{model_name}_s2.pth')
    s1 = os.path.join(models_dir, f'best_{model_name}_s1.pth')
    path = s2 if os.path.exists(s2) else s1
    if not os.path.exists(path):
        raise FileNotFoundError(f'No checkpoint found for {model_name} in {models_dir}')
    model, _, _, _, _ = build_ft_model(model_name, device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    print(f'Loaded {path}')
    return model.cpu()

ft_models = {}
ft_infos  = {}

for backbone_name in ['densenet121', 'densenet169', 'resnext50', 'seresnext50']:
    ft_models[backbone_name] = load_ft_model(backbone_name, MODELS_PATH, device)
    torch.cuda.empty_cache()
    ft_infos[backbone_name]  = {'model_name': backbone_name, 'loaded_from': MODELS_PATH}

print('\nLoaded backbones:')
for k, v in ft_infos.items():
    print(k, v)


In [ ]:
# Collect validation predictions — one backbone on GPU at a time to avoid OOM
ft_val_scores = {}
val_labels_ref = None

for name, model in ft_models.items():
    model = model.to(device)
    scores, labels = get_val_scores(model, val_dataloader, device, tta=True)
    ft_models[name] = model.cpu()
    torch.cuda.empty_cache()
    ft_val_scores[name] = scores
    if val_labels_ref is None:
        val_labels_ref = labels
    auc = metrics.roc_auc_score(labels, scores)
    print(f'{name} TTA val AUC: {auc:.4f}')


In [ ]:
# Ensemble search
def weighted_average_scores(score_dict, weights):
    """
    score_dict: dict model_name -> (N,1) scores
    weights: dict model_name -> weight
    """
    total_w = sum(weights.values())
    out = None
    for name, w in weights.items():
        s = score_dict[name]
        out = s * (w / total_w) if out is None else out + s * (w / total_w)
    return out

def search_ft_ensemble(score_dict, labels, max_models=4):
    model_names = list(score_dict.keys())
    best = {
        'models': None,
        'weights': None,
        'auc': -1.0
    }

    # simple discrete weight grid
    weight_grid = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80]

    for r in range(2, min(max_models, len(model_names)) + 1):
        for combo in itertools.combinations(model_names, r):
            print(f'\nSearching combo: {combo}')

            # generate weight tuples that sum to 1.0 approximately
            if r == 2:
                for w1 in weight_grid:
                    w2 = 1.0 - w1
                    if w2 <= 0:
                        continue
                    weights = {combo[0]: w1, combo[1]: w2}
                    blended = weighted_average_scores(score_dict, weights)
                    auc = metrics.roc_auc_score(labels, blended)
                    if auc > best['auc']:
                        best = {'models': combo, 'weights': weights, 'auc': auc}
                        print('  NEW BEST:', best)

            elif r == 3:
                for w1 in weight_grid:
                    for w2 in weight_grid:
                        w3 = 1.0 - w1 - w2
                        if w3 <= 0:
                            continue
                        weights = {combo[0]: w1, combo[1]: w2, combo[2]: w3}
                        blended = weighted_average_scores(score_dict, weights)
                        auc = metrics.roc_auc_score(labels, blended)
                        if auc > best['auc']:
                            best = {'models': combo, 'weights': weights, 'auc': auc}
                            print('  NEW BEST:', best)

            elif r == 4:
                for w1 in weight_grid:
                    for w2 in weight_grid:
                        for w3 in weight_grid:
                            w4 = 1.0 - w1 - w2 - w3
                            if w4 <= 0:
                                continue
                            weights = {
                                combo[0]: w1,
                                combo[1]: w2,
                                combo[2]: w3,
                                combo[3]: w4
                            }
                            blended = weighted_average_scores(score_dict, weights)
                            auc = metrics.roc_auc_score(labels, blended)
                            if auc > best['auc']:
                                best = {'models': combo, 'weights': weights, 'auc': auc}
                                print('  NEW BEST:', best)

    return best

In [ ]:
# run best ensemble
best_ft_ensemble = search_ft_ensemble(ft_val_scores, val_labels_ref, max_models=4)
print('\nBest FT ensemble found:')
print(best_ft_ensemble)

In [ ]:
# Comparison (single best vs ensemble best)
best_single_name = None
best_single_auc = -1.0

for name, scores in ft_val_scores.items():
    auc = metrics.roc_auc_score(val_labels_ref, scores)
    if auc > best_single_auc:
        best_single_auc = auc
        best_single_name = name

print(f'Best single FT model: {best_single_name}  AUC={best_single_auc:.4f}')
print(f'Best FT ensemble AUC: {best_ft_ensemble["auc"]:.4f}')

In [ ]:
# Test predicitons
def get_test_scores(model, test_dataloader, device, tta=True):
    model.eval()
    n = len(test_dataloader.dataset)
    scores = np.zeros((n, 1), dtype=np.float32)

    cont = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]

            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))

            scores[cont:cont+bs, :] = out.detach().cpu().numpy()
            cont += bs

    return scores

ft_test_scores = {}
for name, model in ft_models.items():
    model = model.to(device)
    ft_test_scores[name] = get_test_scores(model, test_dataloader, device, tta=True)
    ft_models[name] = model.cpu()
    torch.cuda.empty_cache()
    print(f'Computed test scores for {name}')

# Use best val ensemble
best_ft_test_scores = weighted_average_scores(ft_test_scores, best_ft_ensemble['weights'])
np.savetxt(os.path.join(SAVE_PATH, 'ft_ensemble_submission.csv'), best_ft_test_scores, fmt='%.8f')
print(f'Saved: {os.path.join(SAVE_PATH, "ft_ensemble_submission.csv")}')

---
## 10. Generate Test Outputs & Submit

In [ ]:
print('=== Final validation AUC check ===')
auc_custom_final = eval_val_auc(customNetV2, 'CustomNetV2 + SE')
print(f'Best FT ensemble val AUC: {best_ft_ensemble["auc"]:.4f}  '
      f'models={list(best_ft_ensemble["weights"].keys())}')

In [ ]:
outputs_custom = test_model(customNetV2, tta=False)
# best_ft_test_scores already computed above (weighted ensemble, TTA)
outputs_ft = best_ft_test_scores

assert outputs_custom.shape == (1000, 1), f'Expected (1000,1), got {outputs_custom.shape}'
assert outputs_ft.shape     == (1000, 1), f'Expected (1000,1), got {outputs_ft.shape}'
assert np.isfinite(outputs_custom).all(), 'NaN/inf in custom scores'
assert np.isfinite(outputs_ft).all(),     'NaN/inf in ft scores'
print('Shapes and finite-value checks passed.')

custom_csv = os.path.join(SAVE_PATH, 'output_custom.csv')
ft_csv     = os.path.join(SAVE_PATH, 'output_ft.csv')
zip_path   = os.path.join(SAVE_PATH, 'codabench_submission.zip')

with open(custom_csv, mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_custom)
with open(ft_csv, mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_ft)

with ZipFile(zip_path, 'w') as zf:
    zf.write(custom_csv, 'output_custom.csv')
    zf.write(ft_csv,     'output_ft.csv')

print(f'Created: {zip_path}')
print(f'CUSTOM val AUC: {auc_custom_final:.4f}')
print(f'FT     val AUC: {best_ft_ensemble["auc"]:.4f}')
